# 02.6 - Dlib Method D (Letterboxed 96×96) Lip Cropping & 98-Speaker (78 Train / 20 Val) Augmented Dataset Generation (Replaced Videos)
## Tai Phake Visual Speech Recognition (VSR) Project

**Objective**: Construct an end-to-end dataset generation pipeline that extracts sequential video frames from the updated/fixed video directory (`data/experiments/digit_videos_only/`), performs **Dlib 68-landmark mouth localization and Method D letterbox cropping ($96 \times 96$)**, and expands the dataset via **conservative, lip-reading-safe spatiotemporal data augmentation** into exactly **98 speaker directories (78 train + 20 val)**:

- **Source Videos**: `data/experiments/digit_videos_only/` (28 base speakers with updated/fixed videos, excluding outlier speakers `s27` and `s29`).
- **Extracted Raw Frames**: Saved to `data/experiments/extracted_raw_frames_replaced/<speaker>/<digit>/frame_XXXX.jpg`.
- **Dataset Storage**: All 98 speaker directories stored flat under `data/experiments/dlib_aug_cropd_98_78_20_96x96_replaced/`.
- **Speaker Partition**: 22 Base Train (`s1`..`s22` $\to$ 78 folders: 12 spk $\times$ 4 + 10 spk $\times$ 3) and 6 Base Val (`s23, s24, s25, s26, s28, s30` $\to$ 20 folders: 2 spk $\times$ 4 + 4 spk $\times$ 3).
- **80/20 Partition via Symlinks**: Uses `scripts/switch_dataset_80-20.py` to link into `data/processed/train` (78 folders) and `data/processed/val` (20 folders) with zero data leakage.
- **Cropping & Experiment Results**: Geometry logs, summary metrics, manifests, and visualization artifacts are saved to `results/dlib_aug_cropd_98_78_20_96x96_replaced/`.


## 1. Environment, Imports & Centralized Configuration
Initialize core computer vision and augmentation libraries, set deterministic random seeds, and configure centralized paths.

In [1]:
import os
import sys
import re
import json
import time
import shutil
import hashlib
import random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from typing import Dict, List, Tuple, Optional, Any

import cv2
import dlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibility seed
DEFAULT_SEED = 42
random.seed(DEFAULT_SEED)
np.random.seed(DEFAULT_SEED)

# Centralized Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"

VIDEO_INPUT_DIR = DATA_ROOT / "experiments" / "digit_videos_only"
EXTRACTED_RAW_FRAMES_DIR = DATA_ROOT / "experiments" / "extracted_raw_frames_replaced"
DATASET_NAME = "dlib_aug_cropd_98_78_20_96x96_replaced"
OUTPUT_DATASET_DIR = DATA_ROOT / "experiments" / DATASET_NAME
RESULTS_DIR = PROJECT_ROOT / "results" / DATASET_NAME
DLIB_PREDICTOR_PATH = PROJECT_ROOT / "models" / "dlib" / "shape_predictor_68_face_landmarks.dat"
SWITCH_SCRIPT_PATH = PROJECT_ROOT / "scripts" / "switch_dataset_80-20.py"

# Crop & Augmentation Parameters
TARGET_WIDTH = 96
TARGET_HEIGHT = 96
PADDING_X = 15
PADDING_Y = 10
BORDER_VALUE = 0
JPEG_QUALITY = 95

TARGET_TRAIN_FOLDERS = 78
TARGET_VAL_FOLDERS = 20
TOTAL_TARGET_FOLDERS = TARGET_TRAIN_FOLDERS + TARGET_VAL_FOLDERS

print(f"Project Root       : {PROJECT_ROOT}")
print(f"Video Source       : {VIDEO_INPUT_DIR}")
print(f"Extracted Raw Dirs : {EXTRACTED_RAW_FRAMES_DIR}")
print(f"Dataset Output Dir : {OUTPUT_DATASET_DIR}")
print(f"Results Directory  : {RESULTS_DIR}")
print(f"Dlib Predictor     : {DLIB_PREDICTOR_PATH}")

## 2. Zero-Leakage 78/20 Base-Speaker Partition & 98-Folder Expansion

We enforce strict speaker independence matching the optimized champion protocol established in Notebooks 18 & 19:
- **22 Base Train Speakers**: `s1` through `s22`
  - First 12 base speakers (`s1`..`s12`) $\to$ 4 folders each (1 original + 3 augmentations: `sX`, `sX_aug1`, `sX_aug2`, `sX_aug3`) = 48 folders
  - Next 10 base speakers (`s13`..`s22`) $\to$ 3 folders each (1 original + 2 augmentations: `sX`, `sX_aug1`, `sX_aug2`) = 30 folders
  - **Total Train Folders**: $48 + 30 = \mathbf{78}$ folders
- **6 Base Validation Speakers**: `s23`, `s24`, `s25`, `s26`, `s28`, `s30`
  - 2 speakers (`s25`, `s26`) $\to$ 4 folders each (1 original + 3 augmentations) = 8 folders
  - 4 speakers (`s23`, `s24`, `s28`, `s30`) $\to$ 3 folders each (1 original + 2 augmentations) = 12 folders
  - **Total Validation Folders**: $8 + 12 = \mathbf{20}$ folders
- **Total Generated Folders**: Exactly $\mathbf{98}$ folders stored flat in the experiment directory.

In [2]:
def natural_sort_key(s: str) -> int:
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else 0

TRAIN_BASE_SPEAKERS = [f"s{i}" for i in range(1, 23)]
VAL_BASE_SPEAKERS = ["s23", "s24", "s25", "s26", "s28", "s30"]

def build_speaker_folder_mapping(
    base_speakers: List[str],
    target_count: int,
    prefix_aug: str = "aug",
    custom_four_folders: Optional[List[str]] = None
) -> Dict[str, List[str]]:
    n_base = len(base_speakers)
    extra_folders_needed = target_count - n_base
    base_augs_per_spk = extra_folders_needed // n_base
    remainder = extra_folders_needed % n_base

    mapping = {}
    four_folder_set = set(custom_four_folders) if custom_four_folders else set()
    for idx, spk in enumerate(base_speakers):
        dirs = [spk]
        if custom_four_folders is not None:
            num_augs = base_augs_per_spk + (1 if spk in four_folder_set else 0)
        else:
            num_augs = base_augs_per_spk + (1 if idx < remainder else 0)
        for a_idx in range(1, num_augs + 1):
            dirs.append(f"{spk}_{prefix_aug}{a_idx}")
        mapping[spk] = dirs
    return mapping

train_mapping = build_speaker_folder_mapping(TRAIN_BASE_SPEAKERS, TARGET_TRAIN_FOLDERS)
val_mapping = build_speaker_folder_mapping(VAL_BASE_SPEAKERS, TARGET_VAL_FOLDERS, custom_four_folders=["s25", "s26"])

all_train_folders = sorted([f for dirs in train_mapping.values() for f in dirs], key=natural_sort_key)
all_val_folders = sorted([f for dirs in val_mapping.values() for f in dirs], key=natural_sort_key)
all_98_folders = sorted(all_train_folders + all_val_folders, key=natural_sort_key)

print("=" * 80)
print("      ZERO-LEAKAGE 78/20 BASE SPEAKER & FOLDER ALLOCATION VERIFIED")
print("=" * 80)
print(f"Base Train Speakers ({len(TRAIN_BASE_SPEAKERS)}): {TRAIN_BASE_SPEAKERS}")
print(f"Base Val Speakers   ({len(VAL_BASE_SPEAKERS)}): {VAL_BASE_SPEAKERS}")
print(f"Train Folders       ({len(all_train_folders)}): 78 folders")
print(f"Validation Folders  ({len(all_val_folders)}): 20 folders")
print(f"Total Folders       ({len(all_98_folders)}): 98 folders")
print("=" * 80)

## 3. Method D Letterbox Cropping & Dlib Lip ROI Detector
Method D crops facial mouth landmarks tightly (48–67) with fixed padding ($P_x=15\text{ px}, P_y=10\text{ px}$), scales proportionally without aspect distortion, and letterboxes onto a neutral black $96 \times 96$ canvas.

In [3]:
def extract_method_d_crop(
    img: np.ndarray,
    center_x: float,
    center_y: float,
    raw_width: float,
    raw_height: float,
    pad_x: int = PADDING_X,
    pad_y: int = PADDING_Y,
    canvas_size: int = TARGET_WIDTH,
    border_val: int = BORDER_VALUE
) -> Tuple[np.ndarray, int, int, int, int]:
    h_img, w_img = img.shape[:2]
    min_x = center_x - raw_width / 2.0
    max_x = center_x + raw_width / 2.0
    min_y = center_y - raw_height / 2.0
    max_y = center_y + raw_height / 2.0

    x1 = max(0, int(round(min_x - pad_x)))
    y1 = max(0, int(round(min_y - pad_y)))
    x2 = min(w_img, int(round(max_x + pad_x)))
    y2 = min(h_img, int(round(max_y + pad_y)))

    roi = img[y1:y2, x1:x2]
    h_roi, w_roi = roi.shape[:2]
    if h_roi == 0 or w_roi == 0:
        return np.full((canvas_size, canvas_size, 3), border_val, dtype=np.uint8), 0, 0, 0, 0

    scale = canvas_size / max(w_roi, h_roi)
    new_w = min(canvas_size, max(1, int(round(w_roi * scale))))
    new_h = min(canvas_size, max(1, int(round(h_roi * scale))))
    resized_roi = cv2.resize(roi, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((canvas_size, canvas_size, 3), border_val, dtype=np.uint8)
    offset_x = (canvas_size - new_w) // 2
    offset_y = (canvas_size - new_h) // 2
    canvas[offset_y:offset_y + new_h, offset_x:offset_x + new_w] = resized_roi
    return canvas, new_w, new_h, canvas_size - new_w, canvas_size - new_h

print("Method D letterbox cropping function ready.")

## 4. Lip-Reading Safe Data Augmentation
Conservative spatiotemporal transforms ensuring phonetic readability:
1. **Temporally Consistent Geometry**: $\Delta x, \Delta y, \theta, s$ sampled once per variant.
2. **Photometric Realism**: Contrast, brightness, Gaussian sensor noise, and mild Gaussian blur.
3. **Reflection Borders**: `cv2.BORDER_REFLECT_101` eliminates artificial boundary seams.

In [4]:
def get_geometric_rng(base_seed: int, speaker: str, digit: str, variant_idx: int) -> np.random.Generator:
    key = f"{base_seed}_{speaker}_{digit}_aug{variant_idx}_geom".encode("utf-8")
    return np.random.default_rng(int.from_bytes(hashlib.sha256(key).digest()[:4], byteorder="little"))

def get_photometric_rng(base_seed: int, speaker: str, digit: str, frame: str, variant_idx: int) -> np.random.Generator:
    key = f"{base_seed}_{speaker}_{digit}_{frame}_aug{variant_idx}_photo".encode("utf-8")
    return np.random.default_rng(int.from_bytes(hashlib.sha256(key).digest()[:4], byteorder="little"))

def sample_geometric_transform(rng: np.random.Generator, width: int = TARGET_WIDTH, height: int = TARGET_HEIGHT) -> np.ndarray:
    tx = float(rng.uniform(-3.0, 3.0))
    ty = float(rng.uniform(-3.0, 3.0))
    angle = float(rng.uniform(-3.0, 3.0))
    scale = float(rng.uniform(0.97, 1.03))
    M = cv2.getRotationMatrix2D((width / 2.0, height / 2.0), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    return M

def apply_conservative_augmentation(img: np.ndarray, M: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    h, w = img.shape[:2]
    transformed = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    alpha = float(rng.uniform(0.92, 1.08))
    beta = float(rng.uniform(-10.0, 10.0))
    photo = np.clip(alpha * transformed.astype(np.float32) + beta, 0.0, 255.0).astype(np.uint8)
    if rng.random() < 0.50:
        sigma = float(rng.uniform(1.0, 3.5))
        photo = np.clip(photo.astype(np.float32) + rng.normal(0.0, sigma, photo.shape), 0.0, 255.0).astype(np.uint8)
    if rng.random() < 0.30:
        photo = cv2.GaussianBlur(photo, (3, 3), 0.5)
    return photo

print("Lip-reading augmentation functions loaded.")

## 5. Verification of Generated Dataset & Extraction Report
Load and display the execution report from the completed generation pipeline.

In [5]:
summary_path = RESULTS_DIR / "augmentation_summary.json"
if summary_path.exists():
    with open(summary_path, "r") as f:
        summary = json.load(f)
    for k, v in summary.items():
        print(f"{k:30s}: {v}")
else:
    print("Summary file not found at:", summary_path)

## 6. Visual Preview of Generated Method D Crops
Display sample frames across speakers and digits to verify aspect ratio and letterboxing quality.

In [6]:
preview_img_path = RESULTS_DIR / "augmented_samples_preview.png"
if preview_img_path.exists():
    preview_bgr = cv2.imread(str(preview_img_path))
    preview_rgb = cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(15, 12))
    plt.imshow(preview_rgb)
    plt.axis("off")
    plt.title("Tai Phake VSR - Method D 96x96 Preview Grid (Replaced Videos)", fontsize=14)
    plt.show()
else:
    print("Preview image not found at:", preview_img_path)